Copyright 2026 DataRobot, Inc. and its affiliates.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

In [ ]:
from jointfm_client import bootstrap_notebook

bootstrap_notebook(add_src_root=True)

# Skipped Grid Forecast
Two invented stocks, `dragonhoard_gold` and `elvenwood_timber`, trade from Monday to Friday. Their history has no Saturday and no Sunday rows, and a forecast should have none either. This notebook sends such a history as it is: no weekend is filled in, and none is forecast.

A JointFM 1 deployment reads time as positions on one regular grid. With timestamps, the grid step is the smallest gap between two consecutive times of a request, here one day, and every other gap must be a whole multiple of it. A larger gap is not an error. The positions inside it are skipped: Friday and the following Monday sit three positions apart, so the request says that three days passed between the two closes and says nothing about the two days in between. Query times work the same way. They name the positions to forecast and may skip any others.

A skipped grid describes a process that keeps running while nobody observes it. The synthetic prices below behave that way: they move on every calendar day, as news does, and only the weekday closes are recorded. The exchange also closes on one invented holiday, a Wednesday, to show that a gap can have any whole number of steps. The printed counts show how many days lie between consecutive closes.

In [ ]:
import numpy as np
import pandas as pd

HISTORY_SEED = 5
FIRST_MONDAY = "2025-06-02"
LAST_FRIDAY = "2025-10-17"
EXCHANGE_HOLIDAY = "2025-08-13"
TIME_COLUMN = "date"
ASSETS = ["dragonhoard_gold", "elvenwood_timber"]
START_PRICES = [120.0, 45.0]
DAILY_VOLATILITIES = [0.012, 0.018]
CORRELATION = 0.6
SATURDAY = 5
SHOWN_ROWS = 7


def trading_history():
    """Return weekday closes of two correlated prices that move every calendar day."""
    rng = np.random.default_rng(HISTORY_SEED)
    calendar_days = pd.date_range(FIRST_MONDAY, LAST_FRIDAY, freq="D", tz="UTC")
    covariance = [[1.0, CORRELATION], [CORRELATION, 1.0]]
    shocks = rng.multivariate_normal([0.0, 0.0], covariance, size=len(calendar_days))
    log_prices = np.log(START_PRICES) + np.cumsum(shocks * DAILY_VOLATILITIES, axis=0)
    prices = pd.DataFrame(np.exp(log_prices), columns=ASSETS)
    prices.insert(0, TIME_COLUMN, calendar_days)
    is_weekend = prices[TIME_COLUMN].dt.day_of_week >= SATURDAY
    is_holiday = prices[TIME_COLUMN] == pd.Timestamp(EXCHANGE_HOLIDAY, tz="UTC")
    return prices[~(is_weekend | is_holiday)].reset_index(drop=True)


history = trading_history()
days_between_closes = history[TIME_COLUMN].diff().dropna().dt.days
print("days between consecutive closes: how often")
print(days_between_closes.value_counts().sort_index().to_string(header=False))
history.tail(SHOWN_ROWS)

## Sending the trading days as they are
`time_index_mode="absolute_datetime"` with `time_column="date"` sends every row with its timestamp, and the deployment places the rows on the daily grid. Three points need care:

- **Query times follow the same calendar.** The forecast covers the next ten trading days, built with `pandas.bdate_range`. `build_datetime_query_times(...)` continues the last spacing of the history, which suits a history without gaps but would step through the weekend here.
- **The envelope counts grid positions, not rows.** The history may span at most `n_input` positions from its first row to its last, skipped ones included, and the last query time may lie at most `n_output` positions after the last close. `plan_forecast_columns(...)` therefore receives both spans in positions here.
- **Timestamps carry a timezone, and UTC keeps the grid regular.** Stamp a date that has no time of day as midnight UTC, as this notebook does, so that consecutive days are exactly 24 hours apart. Closing times stamped in an exchange's own timezone shift by an hour against that grid whenever daylight saving time changes. Every later row then lies off the grid, and the deployment refuses the request.

In [ ]:
from jointfm_client import JointFMClient, plan_forecast_columns

QUANTILES = (0.1, 0.5, 0.9)
FORECAST_TRADING_DAYS = 10
DAY_LABEL_FORMAT = "%a %Y-%m-%d"

first_close = history[TIME_COLUMN].iloc[0]
last_close = history[TIME_COLUMN].iloc[-1]
QUERY_TIMES = list(
    pd.bdate_range(
        last_close + pd.Timedelta(days=1), periods=FORECAST_TRADING_DAYS, tz="UTC"
    )
)
history_positions = (last_close - first_close).days + 1
positions_ahead = [(query_time - last_close).days for query_time in QUERY_TIMES]

client = JointFMClient.from_env()
instance_ids = [
    entry.instance_id for entry in client.health_instances(cache=True).instances
]
print(f"JointFM instance(s): {instance_ids}")
print(f"{len(history)} history rows span {history_positions} grid positions")
print(
    f"{len(QUERY_TIMES)} query times reach {positions_ahead[-1]} grid positions ahead"
)
plan = plan_forecast_columns(
    health=client.health(cache=True),
    feature_columns=[],
    target_columns=ASSETS,
    history_length=history_positions,
    query_times_length=positions_ahead[-1],
)
result = client.forecast_quantiles(
    history,
    query_times=QUERY_TIMES,
    requested_columns=plan.requested_columns,
    columns=plan.columns,
    time_index_mode="absolute_datetime",
    time_column=TIME_COLUMN,
    quantiles=QUANTILES,
)


def quantile_table(quantile_result):
    """Return one row per query time and one column per asset and quantile."""
    table = quantile_result.to_pandas_tidy().pivot(
        index="query_time", columns=["requested_column", "quantile"], values="value"
    )
    table.columns.names = ["asset", "quantile"]
    return table[ASSETS].reset_index(drop=True)


forecast = quantile_table(result)
if len(forecast) != FORECAST_TRADING_DAYS:
    raise ValueError(
        f"Expected {FORECAST_TRADING_DAYS} forecast rows, got {len(forecast)}"
    )
forecast.index = pd.Index(
    [query_time.strftime(DAY_LABEL_FORMAT) for query_time in QUERY_TIMES],
    name="query time",
)
forecast.insert(0, "positions ahead", positions_ahead)
forecast

## Reading the forecast on the calendar
The chart shows the last closes and the forecast on a calendar axis, one panel per stock because their prices differ in scale. Shaded days are days without a close: neither the history nor the forecast has a value there. Each forecast day carries its median and the interval between its 10% and 90% quantiles. The intervals are drawn day by day instead of as a band, because JointFM 1 answers every position from its own joint distribution, and a band would suggest values on the skipped days.

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

SURFACE = "#fcfcfb"
HISTORY_COLOR = "#52514e"
FORECAST_COLOR = "#2a78d6"
CLOSED_COLOR = "#e1e0d9"
TEXT_PRIMARY = "#0b0b0b"
TEXT_SECONDARY = "#52514e"
TEXT_MUTED = "#898781"
GRIDLINE = "#e1e0d9"
BASELINE = "#c3c2b7"
LINE_WIDTH = 2.0
MARKER_SIZE = 7.0
MARKER_RING_WIDTH = 1.5
INTERVAL_WIDTH = 7.0
INTERVAL_OPACITY = 0.25
CLOSED_OPACITY = 0.6
SHOWN_CLOSES = 15
PANEL_WIDTH_INCHES = 9.0
PANEL_HEIGHT_INCHES = 2.8
HALF_DAY = pd.Timedelta(hours=12)
LOWER_QUANTILE, MEDIAN, UPPER_QUANTILE = QUANTILES

recent = history.tail(SHOWN_CLOSES)
shown_days = pd.date_range(recent[TIME_COLUMN].iloc[0], QUERY_TIMES[-1], freq="D")
closed_days = shown_days.difference(
    pd.DatetimeIndex(recent[TIME_COLUMN]).union(QUERY_TIMES)
)
marker_style = {
    "marker": "o",
    "markersize": MARKER_SIZE,
    "markeredgecolor": SURFACE,
    "markeredgewidth": MARKER_RING_WIDTH,
}


def draw_panel(axis, asset):
    """Draw one stock's recent closes and forecast, and return the legend handles."""
    for closed_day in closed_days:
        axis.axvspan(
            closed_day - HALF_DAY,
            closed_day + HALF_DAY,
            color=CLOSED_COLOR,
            alpha=CLOSED_OPACITY,
            linewidth=0,
        )
    (closes,) = axis.plot(
        recent[TIME_COLUMN],
        recent[asset],
        color=HISTORY_COLOR,
        linewidth=LINE_WIDTH,
        **marker_style,
    )
    intervals = axis.vlines(
        QUERY_TIMES,
        forecast[asset][LOWER_QUANTILE],
        forecast[asset][UPPER_QUANTILE],
        color=FORECAST_COLOR,
        alpha=INTERVAL_OPACITY,
        linewidth=INTERVAL_WIDTH,
        capstyle="butt",
    )
    (medians,) = axis.plot(
        QUERY_TIMES,
        forecast[asset][MEDIAN],
        color=FORECAST_COLOR,
        linestyle="none",
        **marker_style,
    )
    axis.set_facecolor(SURFACE)
    axis.set_title(asset, loc="left", color=TEXT_PRIMARY)
    axis.grid(axis="y", color=GRIDLINE, linewidth=1)
    axis.set_axisbelow(True)
    axis.tick_params(axis="x", colors=TEXT_MUTED, length=0)
    axis.tick_params(axis="y", colors=TEXT_MUTED, length=0)
    for side in ("top", "right", "left"):
        axis.spines[side].set_visible(False)
    axis.spines["bottom"].set_color(BASELINE)
    return [closes, medians, intervals]


figure, axes = plt.subplots(
    len(ASSETS),
    1,
    figsize=(PANEL_WIDTH_INCHES, PANEL_HEIGHT_INCHES * len(ASSETS)),
    sharex=True,
    layout="constrained",
    facecolor=SURFACE,
)
panel_handles = [
    draw_panel(axis, asset) for axis, asset in zip(axes, ASSETS, strict=True)
]
axes[-1].xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%a %b %d"))
legend = figure.legend(
    handles=[
        *panel_handles[0],
        Patch(facecolor=CLOSED_COLOR, alpha=CLOSED_OPACITY, linewidth=0),
    ],
    labels=[
        "close",
        "forecast median",
        f"{LOWER_QUANTILE:.0%} to {UPPER_QUANTILE:.0%} interval",
        "no close",
    ],
    loc="outside lower center",
    ncols=4,
    frameon=False,
)
for text in legend.get_texts():
    text.set_color(TEXT_SECONDARY)
figure.suptitle(
    f"The next {FORECAST_TRADING_DAYS} trading days, weekends skipped",
    color=TEXT_PRIMARY,
)
plt.show()

## The same grid as day numbers
A JointFM 1 deployment places times on its grid and passes only the positions to the model, never the dates. A request that numbers the days itself therefore asks the same question: `time_index_mode="ordinal"` with an integer day counter as the time column, where the counter jumps by three over a weekend. Use this form when your data has no timestamps, or to leave timezones out of the request. The two answers below are identical.

In [ ]:
DAY_COLUMN = "day"

numbered_history = history[ASSETS].assign(
    **{DAY_COLUMN: (history[TIME_COLUMN] - first_close).dt.days}
)
numbered_query_times = [(query_time - first_close).days for query_time in QUERY_TIMES]
numbered = client.forecast_quantiles(
    numbered_history,
    query_times=numbered_query_times,
    requested_columns=plan.requested_columns,
    columns=plan.columns,
    time_index_mode="ordinal",
    time_column=DAY_COLUMN,
    quantiles=QUANTILES,
)
largest_difference = float(np.abs(result.to_numpy() - numbered.to_numpy()).max())
print(f"day numbers of the query times: {numbered_query_times}")
print(f"largest difference between the two answers: {largest_difference}")
numbered_history.tail(SHOWN_ROWS)

## Counting rows instead
The other way to send this history is to count rows: leave out the time column, and every row is one step. Friday and Monday become neighbors on a trading-day clock, and the request no longer says that a weekend lies between them. Both requests are valid, and they describe different processes:

- **Skipped grid:** time runs on while the market is closed. Monday is three positions after Friday, and the ten trading days reach fourteen positions ahead.
- **Row count:** time stands still while the market is closed. Monday is the next step, and the ten trading days reach ten steps ahead.

A row count also takes less of the envelope: the history spans as many positions as it has rows, so more trading days fit into `n_input`, and `n_output` steps reach further in calendar time.

The table compares the width of the interval between the 10% and 90% quantiles under both readings. Which reading suits your data is a question for a backtest: hold out the latest closes and score them under both with `forecast_log_prob(...)`, as `forecast_log_prob.ipynb` shows.

In [ ]:
row_query_times = list(range(len(history), len(history) + FORECAST_TRADING_DAYS))
row_counted = quantile_table(
    client.forecast_quantiles(
        history[ASSETS],
        query_times=row_query_times,
        requested_columns=plan.requested_columns,
        columns=plan.columns,
        quantiles=QUANTILES,
    )
)
row_counted.index = forecast.index


def interval_widths(table):
    """Return the width between the outer quantiles per query time and asset."""
    return pd.DataFrame(
        {
            asset: table[asset][UPPER_QUANTILE] - table[asset][LOWER_QUANTILE]
            for asset in ASSETS
        }
    )


comparison = pd.concat(
    {
        "skipped grid": interval_widths(forecast),
        "row count": interval_widths(row_counted),
    },
    axis=1,
).swaplevel(axis=1)[ASSETS]
comparison.insert(0, "steps ahead", range(1, FORECAST_TRADING_DAYS + 1))
comparison.insert(0, "positions ahead", positions_ahead)
comparison